# CMIP6 core API example

This notebook uses a tiny synthetic CMIP6 dataset to show the public Woodpecker API with a built-in core fix: check a dataset, dry-run the fix, apply it in memory, and re-check the result.

In [ ]:
import numpy as np

import woodpecker
from woodpecker.testing import make_cmip6

Create a deterministic CMIP6-like dataset where near-surface air temperature is stored in Celsius instead of Kelvin.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

dataset

In [ ]:
fix_ids = ["woodpecker.normalize_tas_units_to_kelvin"]

findings = woodpecker.check(dataset, fixes=fix_ids)
findings.fix_ids

A dry run reports what would change but leaves the dataset untouched.

In [ ]:
result = woodpecker.fix(dataset, fixes=findings.fix_ids, dry_run=True)

(
    result.stats,
    result.preview,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values),
)

Apply the same fix in memory with `dry_run=False`.

In [ ]:
write = woodpecker.fix(dataset, fixes=findings.fix_ids, dry_run=False)

(
    write.stats,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values + 273.15),
)

In [ ]:
recheck = woodpecker.check(dataset, fixes=fix_ids)
bool(recheck)